# CamusGPT v3 — Export (adapters → GGUF), Gemma-4-12B

**This notebook does NOT train.** It rebuilds the final model from the two LoRA adapters
already on Drive and exports a q4_k_m GGUF + Ollama Modelfile back to Drive.

**Run all cells.** Every stage is *resumable* (skips if its output already exists) and
*disk-safe* (deletes the previous stage before writing the next), so it is safe to re-run
after a crash — completed stages are detected and skipped.

### Why the previous run died
Four full copies of a 12B model accumulated on the 112 GB local disk:
cache 24 + merged 23.5 + final 23.5 + f16 GGUF 23.5 + q4 7 = **101.5 GB** → out of space
27% into the f16 write. Here each artifact is deleted the moment the next one exists, so
**peak usage is ~48 GB**.

### Inputs (must already be on Drive)
- `adapters/camus3_voice_lora_<corpus>` — Phase 1 (voice); set `PHASE1_CORPUS` below
- `adapters/camus3_guardrail_lora` — Phase 2 (refusals/behaviour)

### Output
- `deploy_v3/camus3-12b.Q4_K_M.gguf` (~7 GB) and `deploy_v3/Modelfile` — Gemma-4 markup,
  taken from `ollama show gemma4:12b --modelfile` rather than hand-written

### Changes from the Gemma-3 export
- Base is `unsloth/gemma-4-12b-it`, loaded with `FastModel` (its model card's loader).
- The Modelfile's `TEMPLATE` / `RENDERER` / `PARSER` lines are **copied verbatim** from
  `ollama show gemma4:12b --modelfile`. That command emits **no `PARAMETER stop` lines at
  all**, because for the gemma4 renderer the turn and channel markup is handled by the
  renderer/parser rather than by stop strings. Inventing stop tokens here would be a
  regression, not a fix.
- Gemma 4 ships a **projector** (`mmproj`) for its multimodal side. We do not need it —
  see the dedicated cell below.

Runtime: any GPU is fine (A100 fastest). Most of the work is disk/CPU bound. ~30–50 min.


In [ ]:
# ── 1. Install (same pins that trained successfully) ─────────────────────────
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.13" peft accelerate bitsandbytes


In [ ]:
# ── 2. Disk helpers + environment report ─────────────────────────────────────
import time
T0 = time.time()

import os, gc, shutil, subprocess, torch

def free_gb(path="/content"):
    return shutil.disk_usage(path).free / 1e9

def used_gb(path):
    if not os.path.exists(path): return 0.0
    if os.path.isfile(path): return os.path.getsize(path) / 1e9
    t = 0
    for r, _d, fs in os.walk(path):
        for f in fs:
            try: t += os.path.getsize(os.path.join(r, f))
            except OSError: pass
    return t / 1e9

def disk(msg=""):
    print(f"   [disk] free {free_gb():6.1f} GB   {msg}")

def need(gb, what):
    """Abort BEFORE a big write if headroom is insufficient."""
    f = free_gb()
    print(f"   [disk] need ~{gb:.1f} GB for {what}; free {f:.1f} GB")
    assert f > gb * 1.15, (
        f"NOT ENOUGH DISK for {what}: need ~{gb:.1f} GB (+15% margin), free {f:.1f} GB. "
        "Delete something under /content or restart the runtime and re-run "
        "(completed stages are skipped).")

def drop(path, label=""):
    """Delete an artifact and report the space reclaimed."""
    if os.path.exists(path):
        sz = used_gb(path)
        shutil.rmtree(path, ignore_errors=True) if os.path.isdir(path) else os.remove(path)
        print(f"   [disk] freed {sz:.1f} GB — removed {label or path}")
    gc.collect(); torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB"
      if torch.cuda.is_available() else "")
disk("at start")


In [ ]:
# ── 3. Mount Drive, config, PRE-FLIGHT ───────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

DRIVE        = "/content/drive/MyDrive/CamusGPT_Training"
BASE_MODEL   = "unsloth/gemma-4-12b-it"

# Must match the Phase-1 run you are exporting (Phase 1 wrote camus3_voice_lora_<corpus>).
PHASE1_CORPUS = "new"
assert PHASE1_CORPUS in ("new", "old"), f"unexpected corpus: {PHASE1_CORPUS}"

VOICE_ADAPTER= f"{DRIVE}/adapters/camus3_voice_lora_{PHASE1_CORPUS}"   # Phase 1
GUARD_ADAPTER= f"{DRIVE}/adapters/camus3_guardrail_lora"               # Phase 2
LOCAL_MERGED = "/content/camus3_sft_merged"                 # stage A output
FINAL        = "/content/camus3_final_hf"                   # stage B output
F16          = "/content/camus3-f16.gguf"                   # stage C1 output
GGUF_NAME    = "camus3-12b.Q4_K_M.gguf"                     # final artifact
Q4           = f"/content/{GGUF_NAME}"                      # stage C2 output
DEPLOY       = f"{DRIVE}/deploy_v3"

missing = []
for name, p in [("voice adapter", VOICE_ADAPTER), ("guardrail adapter", GUARD_ADAPTER)]:
    w = os.path.join(p, "adapter_model.safetensors")
    ok = os.path.exists(w)
    print(f"{'OK  ' if ok else 'MISS'} {name:18s} {p}"
          + (f"   ({os.path.getsize(w)/1e6:.0f} MB)" if ok else ""))
    if not ok: missing.append(name)

if "guardrail adapter" in missing:
    raise SystemExit(
        "\nThe Phase-2 (guardrail) adapter is not on Drive, so there is nothing to export.\n"
        "Check for other names under adapters/:  " + str(sorted(os.listdir(f"{DRIVE}/adapters"))
        if os.path.exists(f"{DRIVE}/adapters") else "adapters/ not found") +
        "\nIf it truly is absent, Phase 2 must be re-run before this notebook can do anything.")
assert "voice adapter" not in missing, "voice adapter missing — Phase 1 output is required"

# The guardrail adapter records the path of the model it was trained on. Stage A must
# recreate that exact path, otherwise Unsloth will not find its base.
import json as _json
_cfg_p = os.path.join(GUARD_ADAPTER, "adapter_config.json")
_recorded = _json.load(open(_cfg_p)).get("base_model_name_or_path", "")
print("\nguardrail adapter was trained on:", _recorded)
if _recorded and _recorded != LOCAL_MERGED and not os.path.isdir(_recorded):
    LOCAL_MERGED = _recorded          # recreate it exactly where the adapter expects it
    print("-> stage A will rebuild that path:", LOCAL_MERGED)

os.makedirs(DEPLOY, exist_ok=True)
disk("after mount")
print("\nStages: A merge voice -> B apply guardrail -> C1 f16 gguf -> C2 q4 -> copy to Drive")


In [ ]:
# ── 4. STAGE A — base + voice adapter -> merged 16-bit  (~23.5 GB, resumable) ─
if os.path.exists(f"{LOCAL_MERGED}/config.json"):
    print(f"SKIP stage A — {LOCAL_MERGED} already exists ({used_gb(LOCAL_MERGED):.1f} GB)")
else:
    need(24 + 23.5, "base download + merged voice model")
    from unsloth import FastModel
    m, t = FastModel.from_pretrained(
        model_name=VOICE_ADAPTER, max_seq_length=2048, dtype=None, load_in_4bit=True)
    m.save_pretrained_merged(LOCAL_MERGED, t, save_method="merged_16bit")
    del m, t; gc.collect(); torch.cuda.empty_cache()
    print(f"stage A done: {used_gb(LOCAL_MERGED):.1f} GB")
disk("after stage A")

# The base weights are cached (~24 GB) but stage B loads from LOCAL_MERGED, not the hub.
drop("/root/.cache/huggingface/hub", "HF hub cache (base weights; re-downloaded only if needed)")
disk("after cache purge")


In [ ]:
# ── 5. STAGE B — merged + guardrail adapter -> FINAL  (~23.5 GB, resumable) ──
if os.path.exists(f"{FINAL}/config.json"):
    print(f"SKIP stage B — {FINAL} already exists ({used_gb(FINAL):.1f} GB)")
else:
    need(23.5, "final merged model")
    from unsloth import FastModel
    m, t = FastModel.from_pretrained(
        model_name=GUARD_ADAPTER, max_seq_length=2048, dtype=None, load_in_4bit=True)
    m.save_pretrained_merged(FINAL, t, save_method="merged_16bit")
    del m, t; gc.collect(); torch.cuda.empty_cache()
    print(f"stage B done: {used_gb(FINAL):.1f} GB")
disk("after stage B")

# FINAL now contains everything; the intermediate is dead weight.
drop(LOCAL_MERGED, "merged voice model (stage A intermediate)")
disk("after dropping stage A output")


In [ ]:
# ── 6. Refresh tokenizer files in FINAL (converter needs clean ones) ─────────
from huggingface_hub import snapshot_download
_tok_files = ["tokenizer.json","tokenizer_config.json","special_tokens_map.json","tokenizer.model"]
src_dir = snapshot_download(BASE_MODEL, allow_patterns=_tok_files)   # a few MB only
copied = []
for f in _tok_files:
    p = os.path.join(src_dir, f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(FINAL, f)); copied.append(f)
print("tokenizer files refreshed from base:", copied)
assert copied, "no tokenizer files found on the base repo — check BASE_MODEL"


In [ ]:
# ── 7. STAGE C1 — FINAL -> f16 GGUF  (~23.5 GB, resumable) ───────────────────
if not os.path.isdir("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !pip install -q -r /content/llama.cpp/requirements.txt

if os.path.exists(F16) and used_gb(F16) > 20:
    print(f"SKIP stage C1 — {F16} already exists ({used_gb(F16):.1f} GB)")
else:
    drop(F16, "partial f16 from an earlier failed run")   # never resume a truncated file
    need(23.5, "f16 GGUF")
    !python /content/llama.cpp/convert_hf_to_gguf.py {FINAL} --outfile {F16} --outtype f16
    assert os.path.exists(F16) and used_gb(F16) > 20, "f16 conversion did not complete"
    print(f"stage C1 done: {used_gb(F16):.1f} GB")
disk("after stage C1")

# Note on the projector: convert_hf_to_gguf.py writes a separate mmproj-*.gguf for
# Gemma 4's multimodal side. We do not need it — CamusGPT is text-only — so the
# converted model must serve standalone. Nothing is deleted here; the projector, if
# written, simply is not copied to Drive and is not referenced by the Modelfile.

# The safetensors are no longer needed — the GGUF carries the weights.
drop(FINAL, "final safetensors (stage B intermediate)")
disk("after dropping stage B output")


In [ ]:
# ── 8. STAGE C2 — quantize f16 -> q4_k_m  (~7 GB, resumable) ─────────────────
QUANT = "/content/llama.cpp/build/bin/llama-quantize"
if os.path.exists(Q4) and used_gb(Q4) > 5:
    print(f"SKIP stage C2 — {Q4} already exists ({used_gb(Q4):.1f} GB)")
else:
    drop(Q4, "partial q4 from an earlier failed run")
    if not os.path.exists(QUANT):
        need(2, "llama.cpp build")
        !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF
        !cmake --build /content/llama.cpp/build --target llama-quantize -j4
    assert os.path.exists(QUANT), "llama-quantize did not build — scroll up for the cmake error"
    need(7, "q4_k_m GGUF")
    !{QUANT} {F16} {Q4} q4_k_m
    assert os.path.exists(Q4) and used_gb(Q4) > 5, "quantization did not complete"
    print(f"stage C2 done: {used_gb(Q4):.1f} GB")
disk("after stage C2")

drop(F16, "f16 GGUF (now superseded by q4)")
disk("after dropping f16")


In [ ]:
# ── 9. Rendering sanity check: the exported GGUF must reproduce training ─────
# The GGUF carries the base chat template, so verify that what the converter embedded
# still renders the way the notebooks trained it: Gemma 4 turn markers, a real system
# turn, and no thought channel (thinking disabled).
from transformers import AutoTokenizer
_t = AutoTokenizer.from_pretrained(BASE_MODEL)
_probe = _t.apply_chat_template(
    [{"role": "system", "content": "SYS_HERE"},
     {"role": "user",   "content": "PROMPT_HERE"},
     {"role": "assistant", "content": "RESPONSE_HERE"}],
    tokenize=False, add_generation_prompt=False, enable_thinking=False)
print(repr(_probe))

assert "<|turn>model\n" in _probe, "exported template lost the model turn marker"
assert "<|turn>user\n" in _probe, "exported template lost the user turn marker"
assert "<|turn>system\n" in _probe, "exported template lost the native system turn"
assert "<turn|>" in _probe, "exported template never closes a turn"
_open, _close = "<|channel>thought\n", "<channel|>"
_pos, _i = _probe.find(_open), 0
while _pos >= 0:
    _s = _pos + len(_open)
    _e = _probe.find(_close, _s)
    assert (_e < 0) or not _probe[_s:_e].strip(), "thinking is not disabled in the export"
    _i, _pos = _i + 1, _probe.find(_open, _s + 1 if _e < 0 else _e)
print(f"rendering verified — {GGUF_NAME} trains and serves the same markup; "
      f"empty thought channel only ({_i} checked)")


### The projector (`mmproj`) is not needed — text-only GGUF

Gemma 4 12B is encoder-free but still **multimodal**: it ships a projector that
`convert_hf_to_gguf.py` writes as a separate `mmproj-*.gguf`, and Ollama's stock
`gemma4:12b` entry carries one (a ~175 MB blob alongside the 7.4 GB model — visible as a
second `FROM` line in `ollama show gemma4:12b --modelfile`). Multimodal use needs that
file loaded next to the main one.

**CamusGPT is text-only, so we deliberately do not use it.** The exported
`camus3-12b.Q4_K_M.gguf` is a single file and the Modelfile references only that file. Do
not "fix" a problem by adding an `mmproj` to the Modelfile — that would be adding an
unused dependency to a text model.

**Verify rather than assume** (this notebook cannot do it for you — Ollama is not on
Colab). After the copy to Drive and download:

```bash
cd rag
ollama create camus3 -f Modelfile.camus3   # single FROM, no mmproj
ollama run camus3 "hey"                    # must answer
```

If the model fails to load or does not stop cleanly with the projector absent, the problem
is in the **export**, not in the Modelfile.


In [ ]:
# ── 10. Copy GGUF to Drive + write the Modelfile from the gemma4 model card ──
SYSTEM_PROMPT = ("You are Albert Camus — writer, journalist, moralist. You speak in the first "
    "person as a man, never as an assistant or program. Your voice is dry, lucid, and "
    "unsentimental, but never cold, rude, or dismissive: you are warmly curious about whoever "
    "you are speaking with, capable of humor and tenderness, and you answer simple or playful "
    "questions simply and gladly rather than deflecting them. You attend to the body, the light, "
    "the sea, and the absurd gap between human longing and the world's silence. You never use "
    "lists, headers, or bullet points. You never break character, never apologize as a servant "
    "does, and never mention being an AI. You speak of your own books in your own words and never "
    "recite or quote passages from them. You respond to what the person actually says and never "
    "invent things they did not mention. You refuse only when truly asked to stop being yourself; "
    "otherwise you engage. You write as Camus would speak — plainly, with restraint and warmth. "
    "What is true of you, and which you never get wrong: you were born in Algeria in 1913 "
    "and raised poor in Belcourt; you won the Nobel Prize in Literature in 1957, no other "
    "field or year; your books are The Stranger, The Plague, The Fall, The Myth of Sisyphus, "
    "The Rebel, Caligula, The Misunderstanding, Exile and the Kingdom, and the unfinished "
    "First Man; Louis Germain was your schoolteacher and Jean Grenier your mentor; you edited "
    "Combat in the Resistance. If someone credits you with a book or deed that is not yours, "
    "you say so plainly rather than playing along. If asked about things from after your time "
    "— machines, devices, words you do not know — you do not pretend to understand them and "
    "you do not help with them; you remain a man of your years. You never invent a specific "
    "you do not remember; you would rather admit the blank.")

# TEMPLATE / RENDERER / PARSER are copied verbatim from:
#     ollama show gemma4:12b --modelfile
# which emits no PARAMETER stop lines — the gemma4 renderer/parser own the turn and
# channel markup. rag/Modelfile.camus3 is the canonical copy of this file; keep them in
# sync. Single FROM, no projector (see the mmproj cell above).
modelfile = "\n".join([
    f"FROM ./{GGUF_NAME}",
    "TEMPLATE {{ .Prompt }}",
    "RENDERER gemma4",
    "PARSER gemma4",
    "PARAMETER temperature 0.45",
    "PARAMETER top_k 40",
    "PARAMETER min_p 0.05",
    "PARAMETER num_ctx 8192",
    "PARAMETER num_predict 1024",
    "PARAMETER repeat_last_n 384",
    "PARAMETER repeat_penalty 1.1",
    'SYSTEM """' + SYSTEM_PROMPT + '"""',
    "",
])
print(modelfile[:400], "\n ...")

need(used_gb(Q4), "copy of the GGUF onto Drive")
dst = f"{DEPLOY}/{GGUF_NAME}"
shutil.copy(Q4, dst)
open(f"{DEPLOY}/Modelfile", "w").write(modelfile)

same = os.path.getsize(dst) == os.path.getsize(Q4)
print(f"gguf copied to Drive: {same}  ({os.path.getsize(dst)/1e9:.2f} GB)")
assert same, "copy size mismatch — re-run this cell"
with open(dst, "rb") as f:
    assert f.read(4) == b"GGUF", "copied file is not a valid GGUF"
print("GGUF magic verified")

drive.flush_and_unmount()
print(f"\nDONE — deploy_v3/ contains {GGUF_NAME} + Modelfile (gemma4 renderer, no "
      "projector, single file).")
print(f"wall-clock elapsed for export: {(time.time() - T0)/60:.1f} min")


## Record the elapsed time, then deploy

The last cell prints the wall-clock figure. **Please write it into `docs/ROADMAP.md`.**
`docs/gemma4_assessment.md` §6 could only give a 2.5–4 h *planning range* for the whole
session because no Gemma-3 run time is recorded anywhere in this repository — no notebook
output, no ROADMAP entry, no log. That gap is the reason it is a range. Add the training
time and this export time to the same entry.

```bash
# from rag/, with camus3-12b.Q4_K_M.gguf beside Modelfile.camus3
cd rag
ollama create camus3 -f Modelfile.camus3
ollama run camus3 "hey"
```

Then point the eval at the new model and compare against the recorded baseline
(factuality 3.5, composite ~3.9):

```bash
GEN_MODEL=camus3 python rag/eval_camus.py     # run twice
```

Note that `rag/camus_rag.py` now sends `"think": false` on every `/api/chat` call, so the
persona does not pay ~2x latency generating reasoning it would discard. Keep that key if
you edit the payloads back.
